# B0 checkpoint verification - Team T69
Inference-only reproduction of the already-trained B0 epoch-2 model. No training, fine-tuning, optimizer, downloads, package installation or weight export. Original artifacts remain read-only. Only validation sample files are opened; the shared manifest's split/scene metadata is read for isolation checks. The original loader receives a validation-only manifest in a new results directory.

Use the original reviewed project, source ZIP, environment JSON and already-extracted verified dataset. Start a fresh matching Colab runtime (recorded Python 3.13.16, PyTorch 2.11.0+cu130, Tesla T4); mount Drive externally. Missing/mismatched prerequisites stop with an exact error. Metric mismatches are saved as FAIL. No CPU or alternative implementation is silently substituted.

## 1. Environment and paths
Edit paths only. Standalone checkpoint is the default; ZIP mode reads only the named checkpoint member into memory, without extraction. Predefined absolute tolerances: RMSE/MAE 1e-6 m, boundary F1 1e-5; relative tolerance zero. They permit small replay differences, not tuning. No results directory is created until source, runtime and validation files pass verification.

In [ ]:
import os, sys, io, json, hashlib, zipfile, platform, subprocess, math, time, uuid
from pathlib import Path
from datetime import datetime, timezone
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
DRIVE_ROOT = Path("/content/drive/MyDrive/TRIDENT-T69")
PROJECT_ROOT = Path("/content/TRIDENT-Depth-T69")
DATA_ROOT = Path("/content/trident-nyu-converted")
MANIFEST = DATA_ROOT / "manifest_development.json"
CHECKPOINT_SOURCE = "file"  # "file" or "zip"; no automatic checkpoint substitution
CHECKPOINT = DRIVE_ROOT / "runs/corrective_v1_B0_dense4_seed69/epoch_002.pt"
CHECKPOINT_ARCHIVE = DRIVE_ROOT / "corrective_v1_B0_dense4_seed69-20261106T163951Z-1-001.zip"
CHECKPOINT_MEMBER = "corrective_v1_B0_dense4_seed69/epoch_002.pt"
SOURCE_ARCHIVE = DRIVE_ROOT / "trident_A1_B0_source.zip"
ENVIRONMENT_JSON = DRIVE_ROOT / "runs/corrective_v1_B0_dense4_seed69_environment.json"
VERIFICATION_ROOT = DRIVE_ROOT / "verification-results"
RESULTS_DIR = VERIFICATION_ROOT / ("B0_epoch002_T69_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid.uuid4().hex[:8])
EXPECTED_CHECKPOINT_SHA256 = "a6416c47a16ec78e97c8b734ae7fb23cba4dbbeb5f5564a3e02d61a4a159c1b2"
EXPECTED_MANIFEST_SHA256 = "d1cdbcf0ff3ee96f2e0c173c3c4701ed3f5c526ab517ce3b988268c3c6a391f5"
EXPECTED_SOURCE_SHA256 = "d8f6a46f17d5195d517afa0e7b43e00d0de7275d80784d6ae3d2fbd6d8b26b35"
EXPECTED_REFERENCE_SHA256 = "bba902eb2f3e9cc5bd66393b3768ffd5777a2d063989e75d2ff0cc762e83b138"
EXPECTED = dict(rmse_m=0.04550963874424752, mae_m=0.012129690590384606, boundary_f1=0.8178656)
TOLERANCES = dict(rmse_m=1e-6, mae_m=1e-6, boundary_f1=1e-5)
CHECKS = []
def require(condition, message):
    if not condition:
        raise RuntimeError("VERIFICATION STOPPED: " + message)
    CHECKS.append(message)
def digest_bytes(value):
    return hashlib.sha256(value).hexdigest()
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()
def need_file(path, description):
    require(Path(path).is_file(), f"Required {description}: {path}")
print("Python:", platform.python_version(), "Executable:", sys.executable)
print("New verification directory:", RESULTS_DIR)


## 2. Checkpoint SHA256 verification
Hash checkpoint bytes before deserialization. Fingerprint the original file/container for a later unchanged-artifact check. ZIP mode rejects missing/duplicate checkpoint members and never loads epoch one.

In [ ]:
require(CHECKPOINT_SOURCE in {"file", "zip"}, "CHECKPOINT_SOURCE must be file or zip")
if CHECKPOINT_SOURCE == "file":
    need_file(CHECKPOINT, "B0 epoch-2 checkpoint")
    require(CHECKPOINT.name == "epoch_002.pt", "Checkpoint filename must be epoch_002.pt")
    ORIGINAL_CONTAINER = CHECKPOINT
    CHECKPOINT_BYTES = CHECKPOINT.read_bytes()
else:
    need_file(CHECKPOINT_ARCHIVE, "supplied B0 archive")
    ORIGINAL_CONTAINER = CHECKPOINT_ARCHIVE
    with zipfile.ZipFile(CHECKPOINT_ARCHIVE) as bundle:
        require(bundle.namelist().count(CHECKPOINT_MEMBER) == 1, "Archive must contain exactly the requested epoch-2 member")
        require(Path(CHECKPOINT_MEMBER).name == "epoch_002.pt", "Archive member must be epoch_002.pt")
        CHECKPOINT_BYTES = bundle.read(CHECKPOINT_MEMBER)
CHECKPOINT_SHA256 = digest_bytes(CHECKPOINT_BYTES)
require(CHECKPOINT_SHA256 == EXPECTED_CHECKPOINT_SHA256, "Checkpoint SHA256 must match expected value")
CONTAINER_SHA256_BEFORE = file_sha256(ORIGINAL_CONTAINER)
print("Verified SHA256:", CHECKPOINT_SHA256)


## 3. Configuration and source verification
Authenticate the source ZIP and external environment JSON using the checkpoint's hash-protected provenance. Match every project Python module and required configuration/reference byte-for-byte before importing project code. Require a fresh kernel to prevent a cached alternative implementation.

Verify original configuration, checkpoint selection/history, package versions, CUDA and GPU. Missing evidence or runtime mismatch stops verification; no package installation occurs. Precision defaults come from a fresh process of the same verified runtime, as used by the original runner, which did not override those defaults.

In [ ]:
import torch, numpy as np, scipy, cv2, PIL, h5py
STATE = torch.load(io.BytesIO(CHECKPOINT_BYTES), map_location="cpu", weights_only=True)
require(isinstance(STATE, dict), "Checkpoint must be a dictionary")
require({"format_version","model_name","model_state","model_configuration","metadata","training_state"} <= STATE.keys(), "Checkpoint must include complete model/configuration/provenance")
META = STATE["metadata"]
require(STATE["format_version"] == 1 and STATE["model_name"] == "residual_cnn_v2", "Checkpoint architecture/format must be residual_cnn_v2 / format 1")
require(META.get("trained") is True and META.get("epochs") == 2, "Checkpoint must be trained for two epochs")
MODEL_CONFIGURATION = dict(upsampling="nearest", alpha=1.0, uncertainty_enabled=False, initialization="exact_zero_v1")
require(STATE["model_configuration"] == MODEL_CONFIGURATION == META.get("model_configuration"), "B0 model configuration: exact_zero_v1, alpha=1, nearest, uncertainty disabled")
require(META.get("uncertainty_available") is False and META.get("boundary_head_supervised") is False, "Original head supervision metadata must match B0")
require(META.get("split_manifest_sha256") == EXPECTED_MANIFEST_SHA256, "Checkpoint must identify the original manifest")
RECORD = META.get("environment_record")
require(isinstance(RECORD, dict) and isinstance(RECORD.get("environment"), dict), "Original source/environment record must be present")
require(RECORD.get("source_id") == EXPECTED_SOURCE_SHA256, "Recorded source identity must match original source ZIP")
need_file(SOURCE_ARCHIVE, "original source ZIP")
require(file_sha256(SOURCE_ARCHIVE) == EXPECTED_SOURCE_SHA256, "Original source ZIP SHA256")
need_file(ENVIRONMENT_JSON, "original B0 environment JSON")
require(file_sha256(ENVIRONMENT_JSON) == META.get("environment_record_sha256"), "Environment JSON SHA256 must match checkpoint")
require(json.loads(ENVIRONMENT_JSON.read_text()) == RECORD, "External and embedded environment records must agree")
require(PROJECT_ROOT.is_dir(), "Original project directory must exist")
EXPERIMENT_PATH = PROJECT_ROOT / "configs/corrective_v1/B0.json"
PIPELINE_PATH = PROJECT_ROOT / "configs/phase5/dense_sr_4x_pipeline.json"
REFERENCE_PATH = PROJECT_ROOT / "docs/pilot_dense4/per_frame.json"
SOURCE_HASHES = {}
with zipfile.ZipFile(SOURCE_ARCHIVE) as bundle:
    names = bundle.namelist()
    require(len(names) == len(set(names)), "Source ZIP must not contain duplicate entries")
    source_names = [n for n in names if n.startswith("trident_depth/") and n.endswith(".py")]
    require(bool(source_names), "Source ZIP must contain the original implementation")
    for name in source_names + ["configs/corrective_v1/B0.json", "configs/phase5/dense_sr_4x_pipeline.json", "docs/pilot_dense4/per_frame.json"]:
        require(name in names and ".." not in Path(name).parts, "Required source archive member: " + name)
        path = PROJECT_ROOT / name
        need_file(path, "original implementation/configuration")
        original = bundle.read(name)
        require(path.read_bytes() == original, "Original source bytes: " + name)
        SOURCE_HASHES[name] = digest_bytes(original)
require(not any(n == "trident_depth" or n.startswith("trident_depth.") for n in sys.modules), "Start a fresh kernel: project modules must not already be cached")
sys.path.insert(0, str(PROJECT_ROOT))
from dataclasses import asdict
from trident_depth.training.corrective import verify_configuration, make_model
from trident_depth.models.checkpoint import load_checkpoint
from trident_depth.training.dataset import TorchDepthDataset
from trident_depth.training.validation import validation_errors
from trident_depth.training.boundaries import boundary_f1_batch, BOUNDARY_EVALUATION
from trident_depth.training.state import seed_everything
CFG, PIPELINE = verify_configuration(EXPERIMENT_PATH)
require(CFG == META.get("config", {}).get("experiment"), "Original B0 experiment must match checkpoint metadata")
require(asdict(PIPELINE) == META.get("config", {}).get("pipeline"), "Original preprocessing must match checkpoint metadata")
require(file_sha256(EXPERIMENT_PATH) == RECORD.get("experiment_sha256"), "Recorded B0 configuration SHA256")
require(file_sha256(PIPELINE_PATH) == RECORD.get("pipeline_sha256"), "Recorded preprocessing SHA256")
require(CFG["experiment_id"] == "B0" and CFG["objective"] == "L_R" and CFG["initializer"] == "bicubic", "B0 must retain its reconstruction objective and bicubic initializer")
require((PIPELINE.protocol,PIPELINE.scale,PIPELINE.target_size) == ("dense_sr",4,None), "Full-frame dense 4x preprocessing")
require(BOUNDARY_EVALUATION == META.get("boundary_evaluation") == RECORD.get("boundary_evaluation"), "Original boundary metric settings must match provenance")
HISTORY = STATE["training_state"].get("history")
require(isinstance(HISTORY,list) and [r.get("epoch") for r in HISTORY] == [1,2], "Embedded history must contain epochs one and two")
require(STATE["training_state"].get("epoch") == 2 and STATE["training_state"].get("best_epoch") == 2, "Saved epoch and best epoch must both be two")
require(min(HISTORY,key=lambda r:r["validation_rmse_m"])["epoch"] == 2, "History must select epoch two by earliest minimum RMSE")
HISTORY_2 = HISTORY[-1]
require(HISTORY_2.get("best_epoch") == 2 and HISTORY_2.get("validation_boundary_frames") == 123, "Epoch-two history must identify best epoch two and 123 validation frames")
require(abs(STATE["training_state"]["best_rmse"]-HISTORY_2["validation_rmse_m"]) < 1e-12, "Best-RMSE state must match embedded history")
RUNTIME = dict(python=platform.python_version(),torch=str(torch.__version__),cuda=torch.version.cuda,
    numpy=np.__version__,scipy=scipy.__version__,opencv=cv2.__version__,pillow=PIL.__version__,h5py=h5py.__version__,
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)
for key in ("python","torch","cuda","numpy","scipy","opencv","pillow","h5py","gpu","gpu_memory_bytes"):
    expected = RECORD["environment"].get(key)
    require(expected is not None and RUNTIME[key] == expected, f"Runtime {key}: expected {expected!r}; found {RUNTIME[key]!r}")
require(torch.cuda.is_available() and META.get("device") == "cuda" and META.get("gpu") == RUNTIME["gpu"], "Original CUDA GPU required; no CPU fallback")
require(META.get("deterministic_algorithms") is True, "Original deterministic inference setting must be recorded")
DEVICE = torch.device("cuda:0")
probe = "import json,torch; print(json.dumps(dict(matmul_tf32=torch.backends.cuda.matmul.allow_tf32,cudnn_tf32=torch.backends.cudnn.allow_tf32,matmul_precision=torch.get_float32_matmul_precision())))"
PRECISION = json.loads(subprocess.check_output([sys.executable,"-c",probe],text=True,env=dict(os.environ,PYTHONDONTWRITEBYTECODE="1")))
torch.set_float32_matmul_precision(PRECISION["matmul_precision"])
torch.backends.cuda.matmul.allow_tf32 = PRECISION["matmul_tf32"]
torch.backends.cudnn.allow_tf32 = PRECISION["cudnn_tf32"]
seed_everything(CFG["seed"])
print("Verified original runtime:",json.dumps(RUNTIME,indent=2))
print("Verified project Python modules:",len(source_names))


## 4. Validation dataset and split verification
Check the original mixed manifest hash, then read only its split/scene metadata and select the existing 123 validation rows in original order. Never call the all-split file-existence audit, extract a full dataset ZIP or inspect non-validation sample paths.

A Python audit hook rejects every dataset file open outside the exact validation allowlist (except shared manifest metadata), blocks dataset directory enumeration, and protects original source/run files from writes. Verify validation RGB/depth/mask hashes only. Create a validation-only manifest in a fresh results directory, changing only these three path fields to absolute paths. All IDs, scenes, units and preprocessing remain original; the original TorchDepthDataset is used unchanged.

In [ ]:
need_file(MANIFEST,"original development manifest")
require(file_sha256(MANIFEST) == EXPECTED_MANIFEST_SHA256 == RECORD.get("manifest_sha256"), "Original development manifest SHA256")
require(file_sha256(REFERENCE_PATH) == EXPECTED_REFERENCE_SHA256 == META.get("validation_reference_sha256"), "Original validation reference SHA256")
ROWS = json.loads(MANIFEST.read_text())  # metadata only, never open non-validation sample files
require(isinstance(ROWS,list), "Manifest must be a list")
VAL_ROWS = [r for r in ROWS if r.get("split") == "validation"]
REFERENCE = json.loads(REFERENCE_PATH.read_text())
require(len(VAL_ROWS) == len(REFERENCE) == 123, "Exactly 123 original validation frames required")
require([r["id"] for r in VAL_ROWS] == [r["id"] for r in REFERENCE], "Original validation IDs and ordering")
VAL_IDS = {r["id"] for r in VAL_ROWS}
LOCKED_IDS = {r["id"] for r in ROWS if r.get("split") == "test" or r.get("official_split") == "test"}
VAL_SCENES = {r["scene_id"] for r in VAL_ROWS}
OTHER_SCENES = {r["scene_id"] for r in ROWS if r.get("split") != "validation"}
require(len(VAL_IDS) == 123 and VAL_IDS.isdisjoint(LOCKED_IDS), "Unique validation IDs isolated from official test labels")
require(len(VAL_SCENES) == 30 and VAL_SCENES.isdisjoint(OTHER_SCENES), "Original 30 validation scenes must be isolated")
require(all(r.get("official_split") == "train" and r.get("depth_scale_to_m") == 1 for r in VAL_ROWS), "Validation origins must be official train; depth units explicitly metres")
def lexical(path):
    return Path(os.path.abspath(os.fspath(path)))
DATA_ROOT, PROJECT_ROOT, RESULTS_DIR = map(lexical,(DATA_ROOT,PROJECT_ROOT,RESULTS_DIR))
ORIGINAL_RUN = lexical(CHECKPOINT.parent) if CHECKPOINT_SOURCE == "file" else None
require(not RESULTS_DIR.is_relative_to(DATA_ROOT), "Results directory must be outside dataset storage")
require(ORIGINAL_RUN is None or not RESULTS_DIR.is_relative_to(ORIGINAL_RUN), "Results directory must be outside original run")
require(not RESULTS_DIR.exists(), "Results directory must not exist")
ALLOWLIST = {lexical(DATA_ROOT/r[key]) for r in VAL_ROWS for key in ("rgb","depth","mask")}
require(len(ALLOWLIST) == 369 and all(p.is_relative_to(DATA_ROOT) for p in ALLOWLIST), "369 distinct validation files must remain inside dataset storage")
DATA_OPEN_LOG, DENIED_ACCESS = [], []
PROTECTED_FILES = {lexical(p) for p in (ORIGINAL_CONTAINER,SOURCE_ARCHIVE,ENVIRONMENT_JSON,MANIFEST)}
MANIFEST_LEXICAL = lexical(MANIFEST)
def access_guard(event,args):
    if event not in {"open","os.listdir","os.scandir","os.remove","os.rmdir","os.mkdir","os.rename"}:
        return
    for value in (args[:2] if event == "os.rename" else args[:1]):
        if not isinstance(value,(str,bytes,os.PathLike)):
            continue
        path = lexical(os.fsdecode(value))
        in_data = path.is_relative_to(DATA_ROOT)
        protected = path in PROTECTED_FILES or path.is_relative_to(PROJECT_ROOT) or (ORIGINAL_RUN is not None and path.is_relative_to(ORIGINAL_RUN))
        if event == "open":
            mode, flags = args[1] or "", args[2] or 0
            writes = any(c in str(mode) for c in "wax+") or bool(flags & (os.O_WRONLY|os.O_RDWR|os.O_CREAT|os.O_TRUNC|os.O_APPEND))
            if in_data and (path not in ALLOWLIST|{MANIFEST_LEXICAL} or writes):
                DENIED_ACCESS.append(dict(event=event,path=str(path)))
                raise PermissionError("Non-validation dataset access denied: "+str(path))
            if writes and protected and not path.is_relative_to(RESULTS_DIR):
                raise PermissionError("Original artifact write denied: "+str(path))
            if in_data:
                DATA_OPEN_LOG.append(dict(path=str(path),operation="read"))
        elif in_data:
            DENIED_ACCESS.append(dict(event=event,path=str(path)))
            raise PermissionError("Dataset enumeration/mutation denied: "+str(path))
        elif event in {"os.remove","os.rmdir","os.mkdir","os.rename"} and protected and not path.is_relative_to(RESULTS_DIR):
            raise PermissionError("Original artifact mutation denied: "+str(path))
sys.addaudithook(access_guard)
VALIDATION_PROVENANCE, PROJECTED_ROWS = [], []
for row,ref in zip(VAL_ROWS,REFERENCE):
    require(row["scene_id"] == ref["scene_id"], "Original validation scene: "+row["id"])
    projected, hashes = dict(row), {}
    for key in ("rgb","depth","mask"):
        path = lexical(DATA_ROOT/row[key])
        require(path.resolve() == path, "Validation file must not be a symlink alias: "+str(path))
        need_file(path,"validation "+key)
        hashes[key] = file_sha256(path)
        require(hashes[key] == ref["source_files_sha256"][key], "Original validation file SHA256: "+row["id"]+"/"+key)
        projected[key] = str(path)
    PROJECTED_ROWS.append(projected)
    VALIDATION_PROVENANCE.append(dict(id=row["id"],scene_id=row["scene_id"],source_files_sha256=hashes,original_mask_sha256=ref["mask_sha256"]))
RESULTS_DIR.mkdir(parents=True,exist_ok=False)
VALIDATION_MANIFEST = RESULTS_DIR/"validation_only_manifest.json"
with VALIDATION_MANIFEST.open("x",encoding="utf-8") as stream:
    json.dump(PROJECTED_ROWS,stream,indent=2)
DATASET = TorchDepthDataset(VALIDATION_MANIFEST,PIPELINE,"validation",initializer=CFG["initializer"])
require(len(DATASET) == 123 and [r["id"] for r in DATASET.source.records] == [r["id"] for r in VAL_ROWS], "Original loader must expose only the 123 original validation rows")
print("Validation-only loader ready; no train/test sample files opened.")


## 5. Model reconstruction and checkpoint loading
Use the original load_checkpoint, whose model registry reconstructs the model and loads with strict=True. It accepts verified checkpoint bytes through BytesIO. Independently call the original make_model and explicit strict loading to check keys and shapes. Fresh zero initialization is always overwritten by the trained tensors before inference. No alternative architecture or FastAPI hybrid path is substituted.

In [ ]:
MODEL,loaded_metadata = load_checkpoint(io.BytesIO(CHECKPOINT_BYTES),require_trained=True,device=DEVICE)
require(loaded_metadata == META, "Original checkpoint loader must preserve metadata")
RECONSTRUCTED = make_model(CFG)
expected_state = RECONSTRUCTED.state_dict()
missing_keys = sorted(set(expected_state)-set(STATE["model_state"]))
unexpected_keys = sorted(set(STATE["model_state"])-set(expected_state))
require(not missing_keys and not unexpected_keys, f"Strict checkpoint keys: missing={missing_keys}, unexpected={unexpected_keys}")
require(all(expected_state[k].shape == STATE["model_state"][k].shape for k in expected_state), "Checkpoint tensor shapes must match original constructor")
strict_result = RECONSTRUCTED.load_state_dict(STATE["model_state"],strict=True)
require(not strict_result.missing_keys and not strict_result.unexpected_keys, "Strict load must report no missing/unexpected keys")
require(sum(p.numel() for p in MODEL.parameters()) == 473059, "Original B0 architecture must have 473059 parameters")
require(MODEL.configuration() == MODEL_CONFIGURATION and MODEL.uncertainty_enabled is False, "Reconstruction must preserve original B0 configuration")
require(all(p.dtype == torch.float32 for p in MODEL.parameters()), "All model parameters must be FP32")
require(all(torch.equal(v.detach().cpu(),STATE["model_state"][k]) for k,v in MODEL.state_dict().items()), "Every loaded tensor must equal the checkpoint; no head reset after loading")
MODEL.eval()
MODEL.requires_grad_(False)
require(not MODEL.training, "Inference must use eval mode")
MODEL_HASH_BEFORE = {k:digest_bytes(v.detach().cpu().contiguous().numpy().tobytes()) for k,v in MODEL.state_dict().items()}
del RECONSTRUCTED, expected_state
print("Original loader and explicit strict reconstruction passed; missing/unexpected keys: [] / []")


## 6. Validation inference
The unchanged TorchDepthDataset performs metre conversion/validity cleanup, RGB normalization, 4x area degradation, bicubic initialization and nearest support resizing. Call the original model with the same tensors as the original validation runner. No FastAPI hybrid fusion or measurement projection is applied. Use FP32, eval mode, inference_mode and disabled autocast. Aggregate original metric functions online; full prediction maps are not written.

In [ ]:
from torch.utils.data import DataLoader
LOADER = DataLoader(DATASET,batch_size=1,shuffle=False,num_workers=0)
REFERENCE_BY_ID = {r["id"]:r for r in REFERENCE}
FRAME_RESULTS, LOADED_IDS, BOUNDARY_SCORES = [], [], []
SUM_SQUARED, SUM_ABSOLUTE, VALID_PIXELS = 0.0, 0.0, 0
STARTED = time.perf_counter()
with torch.inference_mode(), torch.autocast(device_type="cuda",enabled=False):
    for batch in LOADER:
        sample_id = batch["sample_id"][0]
        require(sample_id in VAL_IDS and sample_id not in LOCKED_IDS, "Inference must be validation-only: "+sample_id)
        require(sample_id not in LOADED_IDS, "Frame must not be evaluated twice: "+sample_id)
        ref = REFERENCE_BY_ID[sample_id]
        require(batch["rgb_present"].all().item(), "Original RGB guidance required: "+sample_id)
        require(list(batch["target"].shape) == [1,1,480,640] and list(batch["observations"].shape) == [1,1,120,160], "Original full-frame dense 4x dimensions: "+sample_id)
        mask_hash = digest_bytes(batch["target_valid"].contiguous().numpy().tobytes())
        require(mask_hash == ref["mask_sha256"] and int(batch["target_valid"].sum()) == ref["valid_pixels"], "Original evaluation mask and valid-pixel count: "+sample_id)
        output = MODEL(batch["rgb"].to(DEVICE),batch["initial"].to(DEVICE),batch["observed_support"].to(DEVICE),PIPELINE.scale,PIPELINE.protocol)
        require("log_scale" not in output, "B0 must not execute/expose uncertainty")
        prediction = output["depth"]
        require(prediction.dtype == torch.float32 and torch.isfinite(prediction).all().item(), "Finite FP32 metric depth required: "+sample_id)
        squared,absolute,count = validation_errors(prediction,batch["target"].to(DEVICE),batch["target_valid"].to(DEVICE))
        scores = boundary_f1_batch(prediction,batch["target"],batch["target_valid"])
        require(count > 0 and len(scores) == 1 and math.isfinite(scores[0]), "Original metric implementation must produce finite frame results: "+sample_id)
        SUM_SQUARED += squared
        SUM_ABSOLUTE += absolute
        VALID_PIXELS += count
        BOUNDARY_SCORES.extend(scores)
        LOADED_IDS.append(sample_id)
        FRAME_RESULTS.append(dict(id=sample_id,scene_id=ref["scene_id"],mask_sha256=mask_hash,
            sum_squared_error_m2=squared,sum_absolute_error_m=absolute,pixels=count,rmse_m=math.sqrt(squared/count),mae_m=absolute/count,
            boundary_f1=scores[0],prediction_sha256=digest_bytes(prediction.detach().cpu().contiguous().numpy().tobytes())))
        if len(LOADED_IDS) % 20 == 0:
            print(f"Verified {len(LOADED_IDS)}/123 frames")
INFERENCE_SECONDS = time.perf_counter()-STARTED
require(LOADED_IDS == [r["id"] for r in VAL_ROWS], "All 123 original validation frames must be evaluated exactly once in original order")
print("Inference complete:",len(LOADED_IDS),"validation frames;",VALID_PIXELS,"valid pixels")


## 7. Metric calculation and comparison
Use sqrt(total squared error / total valid pixels) for global RMSE and total absolute error / total valid pixels for MAE, both in metres. Do not average frame RMSEs. Boundary F1 is the mean of the original 123 per-frame scores. Compare to supplied targets and full-precision checkpoint history using the tolerances declared before inference. Save discrepancies as FAIL without hiding them or changing tolerance.

In [ ]:
require(len(LOADED_IDS) == len(BOUNDARY_SCORES) == 123 and VALID_PIXELS > 0, "Aggregation requires all original validation frames")
REPRODUCED = dict(rmse_m=float(np.sqrt(SUM_SQUARED/VALID_PIXELS)),mae_m=SUM_ABSOLUTE/VALID_PIXELS,boundary_f1=float(np.mean(BOUNDARY_SCORES)))
RECORDED = dict(rmse_m=HISTORY_2["validation_rmse_m"],mae_m=HISTORY_2["validation_mae_m"],boundary_f1=HISTORY_2["validation_boundary_f1"])
def compare(reference):
    return {key:dict(expected=reference[key],reproduced=value,signed_difference=value-reference[key],
        absolute_difference=abs(value-reference[key]),absolute_tolerance=TOLERANCES[key],relative_tolerance=0.0,
        passed=bool(math.isfinite(value) and abs(value-reference[key]) <= TOLERANCES[key])) for key,value in REPRODUCED.items()}
COMPARISON, HISTORY_COMPARISON = compare(EXPECTED), compare(RECORDED)
REPRODUCTION_PASSED = all(r["passed"] for r in list(COMPARISON.values())+list(HISTORY_COMPARISON.values()))
print(json.dumps(dict(reproduced=REPRODUCED,expected_comparison=COMPARISON,checkpoint_history_comparison=HISTORY_COMPARISON,reproduction_passed=REPRODUCTION_PASSED),indent=2))


## 8. Test-split isolation audit
Audit actual evaluated IDs and Python dataset file-open events. The guard covers the inspected original NumPy/Pillow loader's Python file access; this is not an OS-wide trace. Shared manifest metadata was read for isolation checks only. No all-split file audit or dataset archive access occurs. The guard remains active through report saving.

In [ ]:
require(set(LOADED_IDS) == VAL_IDS and set(LOADED_IDS).isdisjoint(LOCKED_IDS), "Actual inference IDs must contain only validation, with zero test IDs")
require(not DENIED_ACCESS, "No forbidden dataset access may have been attempted")
require(all(Path(r["path"]) in ALLOWLIST|{MANIFEST_LEXICAL} for r in DATA_OPEN_LOG), "Every dataset open must be a validation file or shared manifest metadata")
require(not any(p.grad is not None for p in MODEL.parameters()), "No parameter gradients may have been created")
require(all(digest_bytes(v.detach().cpu().contiguous().numpy().tobytes()) == MODEL_HASH_BEFORE[k] for k,v in MODEL.state_dict().items()), "Model tensors unchanged after inference")
require(file_sha256(ORIGINAL_CONTAINER) == CONTAINER_SHA256_BEFORE, "Original checkpoint container unchanged")
require(digest_bytes(CHECKPOINT_BYTES) == EXPECTED_CHECKPOINT_SHA256, "Verified checkpoint bytes unchanged")
require(file_sha256(SOURCE_ARCHIVE) == EXPECTED_SOURCE_SHA256, "Original source archive unchanged")
require(file_sha256(ENVIRONMENT_JSON) == META["environment_record_sha256"], "Original environment record unchanged")
require(file_sha256(MANIFEST) == EXPECTED_MANIFEST_SHA256, "Original manifest unchanged")
for name,expected_hash in SOURCE_HASHES.items():
    require(file_sha256(PROJECT_ROOT/name) == expected_hash, "Original source/configuration/reference unchanged: "+name)
ISOLATION = dict(shared_manifest_split_scene_metadata_read=True,evaluated_split="validation",validation_frames_loaded=len(LOADED_IDS),
    training_samples_loaded=0,official_test_samples_loaded=0,official_test_sample_files_opened=0,
    forbidden_access_attempts=DENIED_ACCESS,dataset_open_events=len(DATA_OPEN_LOG),
    distinct_validation_files_opened=len({r["path"] for r in DATA_OPEN_LOG if Path(r["path"]) in ALLOWLIST}),
    audit_scope="Python open/listdir/scandir events of original loader; not OS-wide tracing",full_dataset_archive_accessed=False,
    original_artifacts_unchanged=True)
print(json.dumps(ISOLATION,indent=2))


## 9. Save verification reports
Exclusive file creation prevents overwriting artifacts or earlier reports. JSON includes full configurations/source/runtime provenance, sufficient statistics, original masks, per-frame prediction hashes, numerical differences/tolerances and file-open isolation evidence. Markdown summarizes the measured result. Metric mismatches are explicitly reported as FAIL. Prerequisite failures stop in their cell with the missing/mismatched requirement and do not produce a false success report. Re-execution requires a fresh kernel and creates a new directory.

In [ ]:
STATUS = "PASS" if REPRODUCTION_PASSED else "FAIL"
REPORT = dict(status=STATUS,purpose="Independent inference-only B0 epoch-2 validation reproduction",created_utc=datetime.now(timezone.utc).isoformat(),
    checkpoint_sha256=CHECKPOINT_SHA256,original_container=str(ORIGINAL_CONTAINER),original_container_sha256=CONTAINER_SHA256_BEFORE,
    checkpoint_member=CHECKPOINT_MEMBER if CHECKPOINT_SOURCE == "zip" else None,model_name=STATE["model_name"],parameters=473059,
    model_configuration=MODEL_CONFIGURATION,best_epoch=STATE["training_state"]["best_epoch"],experiment=CFG,pipeline=asdict(PIPELINE),
    original_manifest_sha256=EXPECTED_MANIFEST_SHA256,validation_manifest_sha256=file_sha256(VALIDATION_MANIFEST),
    original_reference_sha256=EXPECTED_REFERENCE_SHA256,source_archive_sha256=EXPECTED_SOURCE_SHA256,verified_source_files_sha256=SOURCE_HASHES,
    environment_record=RECORD,environment_record_sha256=file_sha256(ENVIRONMENT_JSON),runtime=RUNTIME,precision_settings=PRECISION,
    inference_settings=dict(eval_mode=True,inference_mode=True,autocast=False,parameter_dtype="float32",batch_size=1,shuffle=False,num_workers=0,
        deterministic_algorithms=torch.are_deterministic_algorithms_enabled(),cudnn_benchmark=torch.backends.cudnn.benchmark,cudnn_deterministic=torch.backends.cudnn.deterministic),
    metric_aggregation=dict(rmse="sqrt(sum_squared_error_m2 / valid_pixels)",mae="sum_absolute_error_m / valid_pixels",boundary=BOUNDARY_EVALUATION),
    sum_squared_error_m2=SUM_SQUARED,sum_absolute_error_m=SUM_ABSOLUTE,valid_pixels=VALID_PIXELS,validation_frames=123,
    reproduced=REPRODUCED,expected=EXPECTED,tolerances=TOLERANCES,expected_comparison=COMPARISON,checkpoint_history_comparison=HISTORY_COMPARISON,
    reproduction_passed=REPRODUCTION_PASSED,inference_seconds=INFERENCE_SECONDS,isolation=ISOLATION,dataset_open_ledger=DATA_OPEN_LOG,
    validation_provenance=VALIDATION_PROVENANCE,per_frame=FRAME_RESULTS,strict_load=dict(missing_keys=[],unexpected_keys=[]),
    optimizer_steps=0,training_or_finetuning_performed=False,checkpoint_or_weight_export_performed=False,original_checkpoint_unchanged=True,completed_checks=CHECKS)
JSON_REPORT, MD_REPORT = RESULTS_DIR/"B0_epoch002_verification.json", RESULTS_DIR/"B0_epoch002_verification.md"
require(not JSON_REPORT.exists() and not MD_REPORT.exists(), "Both report files must be new")
lines = ["# B0 checkpoint verification - Team T69","","Status: **"+STATUS+"**","","Checkpoint SHA256: `"+CHECKPOINT_SHA256+"`","",
    "Best epoch: 2; residual_cnn_v2; 473,059 parameters. 123 original validation frames; errors in metres.","",
    "| Metric | Expected | Reproduced | Signed difference | Absolute tolerance | Pass |","| --- | ---: | ---: | ---: | ---: | --- |"]
for key,row in COMPARISON.items():
    lines.append(f"| {key} | {row['expected']:.15g} | {row['reproduced']:.15g} | {row['signed_difference']:+.8g} | {row['absolute_tolerance']:.1g} | {row['passed']} |")
lines += ["","Both supplied targets and full-precision checkpoint history use the predefined tolerances.","",
    "Global pixel-weighted RMSE/MAE; mean per-frame boundary F1 at 0.05 m / 1 pixel tolerance.","",
    "Source/configuration and evaluation-mask hashes verified. Model tensors and original checkpoint bytes unchanged.","",
    "No training, optimizer steps, fine-tuning, export or official test sample-file access occurred.",
    "Shared manifest split/scene metadata was read only for isolation; the Python file guard allowed validation sample files only.","",
    "Runtime: `"+json.dumps(RUNTIME,sort_keys=True)+"`","",f"Inference time: {INFERENCE_SECONDS:.3f} seconds; valid pixels: {VALID_PIXELS}.","",
    "JSON includes full configuration/provenance, signed and absolute differences against both references, per-frame statistics/hashes and the dataset-open ledger.","",
    "This is numerical reproduction evidence, not test-set generalization or a new training result."]
with JSON_REPORT.open("x",encoding="utf-8") as stream:
    json.dump(REPORT,stream,indent=2,allow_nan=False)
with MD_REPORT.open("x",encoding="utf-8") as stream:
    stream.write("\n".join(lines)+"\n")
print(STATUS,"JSON:",JSON_REPORT,"Markdown:",MD_REPORT)
